# 02 — LeRobot: Collect, Train, Deploy

**Purpose**: Full LeRobot coaching workflow — provision the MI100 node with Ansible, configure the Pi for data collection, train an ACT policy, and fetch the checkpoint back to the edge.

**Prereqs**:
- `01_reserve_node` completed — `ansible/inventory.ini` exists
- `.env` populated with `HF_USER`, `HF_TOKEN`, `DATASET_REPO_ID`
- Pi connected with leader/follower arms and cameras

**Outcome**: Trained ACT checkpoint at `MODEL_REPO_ID` on HuggingFace Hub; benchmark timing saved to `bench/results/`.

---

In [ ]:
import os
import time
import json
import shlex
import subprocess
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from tqdm.notebook import tqdm

load_dotenv(dotenv_path=Path('..') / '.env', override=False)

# Benchmark: record start time
_bench = {
    "notebook": "02_lerobot",
    "started_at": datetime.utcnow().isoformat(),
    "timings": {},
    "config": {}
}
_t0 = time.monotonic()

FLOATING_IP  = os.getenv("CONTROL_FLOATING_IP")
HF_USER      = os.getenv("HF_USER")
HF_TOKEN     = os.getenv("HF_TOKEN")
DATASET_REPO = os.getenv("DATASET_REPO_ID", f"{HF_USER}/soarm101-pick-block")
MODEL_REPO   = os.getenv("MODEL_REPO_ID", f"{HF_USER}/act-pick-block")
POLICY       = os.getenv("POLICY", "act")
PI_HOST      = os.getenv("PI_HOST", "192.168.4.191")
PI_PORT      = os.getenv("PI_PORT", "22222")
PI_USER      = "root"
PI_IMAGE     = f"{HF_USER}/lerobot-soarm101:latest"

_bench["config"] = {
    "dataset": DATASET_REPO, "model": MODEL_REPO,
    "policy": POLICY, "node_ip": FLOATING_IP,
}

print(f"Node:    {FLOATING_IP}")
print(f"Dataset: {DATASET_REPO}")
print(f"Model:   {MODEL_REPO}")
print(f"Policy:  {POLICY}")
print(f"Pi:      {PI_HOST}:{PI_PORT}")

## 1. Provision Training Node with Ansible

Installs ROCm 6.3, Miniconda, and LeRobot. Idempotent — safe to re-run (skips completed steps).
Takes ~15–20 min on first run, ~2 min on subsequent runs.

In [ ]:
if not FLOATING_IP or FLOATING_IP == "REPLACE_ME_AFTER_PROVISION":
    raise RuntimeError("CONTROL_FLOATING_IP not set in .env — run 01_reserve_node first.")

_t_provision = time.monotonic()
ansible_dir = Path("..") / "ansible"
inventory   = ansible_dir / "inventory.ini"
playbook    = ansible_dir / "playbooks" / "setup_training_node.yml"

cmd = [
    "ansible-playbook",
    "-i", str(inventory),
    str(playbook),
    "-v",
]

print(f"Running: {' '.join(cmd)}")
print("=" * 60)

proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, cwd=str(ansible_dir)
)
for line in proc.stdout:
    print(line, end="")

rc = proc.wait()
print("=" * 60)
if rc != 0:
    raise RuntimeError(f"Ansible failed (exit {rc}) — re-run to retry skipped steps.")
print("Provisioning complete.")

_bench["timings"]["provision_s"] = round(time.monotonic() - _t_provision, 1)
print(f"  Took {_bench['timings']['provision_s']}s")

## 2. Verify GPU on Training Node

In [ ]:
from chi import ssh

with ssh.Remote(FLOATING_IP) as conn:
    print("=== ROCm ===")
    conn.run("rocm-smi --version 2>/dev/null || echo 'ROCm not yet installed'")
    print("\n=== GPU ===")
    conn.run("rocminfo 2>/dev/null | grep -A2 'gfx908' | head -6 || lspci | grep -i amd")
    print("\n=== PyTorch ===")
    conn.run(
        "source ~/miniconda3/bin/activate lerobot 2>/dev/null && "
        "python -c \"import torch; print(torch.__version__, torch.cuda.is_available())\""
        " || echo 'lerobot env not ready yet'"
    )

## 3. Verify Pi Devices

Check serial ports (leader/follower arms) and cameras before coaching.

In [ ]:
def pi_ssh(cmd, capture=False):
    """Run a command on the Pi over SSH."""
    ssh_cmd = f"ssh -p {PI_PORT} -o StrictHostKeyChecking=no {PI_USER}@{PI_HOST}"
    full = shlex.split(ssh_cmd) + ["bash", "-c", cmd]
    if capture:
        return subprocess.check_output(full, text=True).strip()
    else:
        subprocess.run(full, check=True)

print("=== Serial Ports ===")
pi_ssh("ls /dev/ttyACM* 2>/dev/null || echo 'NO SERIAL PORTS'")

print("\n=== Cameras ===")
pi_ssh("ls /dev/video0 /dev/video2 2>/dev/null && echo 'cameras OK' || echo 'cameras NOT FOUND'")

print("\n=== Running Containers ===")
pi_ssh("balena ps --format 'table {{.Names}}\t{{.Status}}' 2>/dev/null || docker ps --format 'table {{.Names}}\t{{.Status}}'")

## 4. Configure Collection Run

In [ ]:
ROBOT        = "alpha"          # robot name from fleet.yaml
DATASET_SLUG = "pick-block"     # becomes {HF_USER}/soarm101-{slug}
TASK_DESC    = "Pick up the block and place it in the bowl"
NUM_EPISODES = 50
EPISODE_TIME = 30   # seconds per episode
RESET_TIME   = 10   # seconds to return to home between episodes

total_s = NUM_EPISODES * (EPISODE_TIME + RESET_TIME) - RESET_TIME

print(f"Dataset:  {DATASET_REPO}")
print(f"Episodes: {NUM_EPISODES} x {EPISODE_TIME}s + {RESET_TIME}s reset")
print(f"Total:    ~{total_s}s ({total_s//60}m {total_s%60}s)")
print()
print("Run in your terminal to start coaching:")
print()
print(f"ssh -p {PI_PORT} -t {PI_USER}@{PI_HOST} \\")
print(f'  "balena run -it --privileged \\')
print(f'   --device=/dev/ttyACM0 --device=/dev/ttyACM1 \\')
print(f'   --device=/dev/video0 --device=/dev/video2 \\')
print(f'   -v /tmp/fleet.yaml:/app/config/fleet.yaml \\')
print(f'   -v /mnt/data/calibration:/app/calibration \\')
print(f'   -v /mnt/data/datasets:/app/data \\')
print(f'   {PI_IMAGE} \\')
print(f'   coachable --fleet /app/config/fleet.yaml collect \\')
print(f'     --robot {ROBOT} --dataset {DATASET_SLUG} \\')
print(f'     --episodes {NUM_EPISODES} --episode-time {EPISODE_TIME} --reset-time {RESET_TIME} \\')
print(f"     --task '{TASK_DESC}' --no-push\"")

## 5. Push Dataset to HuggingFace Hub

Run after coaching is complete and you've verified at least one episode via replay.

In [ ]:
push_cmd = (
    f"balena run --privileged "
    f"-v /mnt/data/datasets:/app/data "
    f"-e HF_TOKEN={HF_TOKEN} "
    f"{PI_IMAGE} "
    f"huggingface-cli upload {DATASET_REPO} /app/data/{HF_USER}/soarm101-{DATASET_SLUG} --repo-type dataset"
)

print("Push dataset from Pi:")
print(f"  ssh -p {PI_PORT} {PI_USER}@{PI_HOST}")
print(f"  {push_cmd}")
print()
print(f"Dataset will be at: https://huggingface.co/datasets/{DATASET_REPO}")

## 6. Launch Training on MI100

In [ ]:
_t_train = time.monotonic()

# MI100 constraints: ACT recommended (no Flash Attention dependency)
# Pi0 workarounds: add --policy.gradient_checkpointing=true --policy.dtype=bfloat16 --batch_size=8
train_cmd = (
    f"source ~/miniconda3/bin/activate lerobot && "
    f"cd ~/lerobot && "
    f"python lerobot/scripts/train.py "
    f"--dataset.repo_id={DATASET_REPO} "
    f"--policy.path=lerobot/{POLICY} "
    f"--output_dir=outputs/train/{POLICY}_{DATASET_SLUG} "
    f"--job_name={POLICY}_{DATASET_SLUG} "
    f"--policy.device=cuda "
    f"--wandb.enable=false"
)

# Launch in a tmux session so it survives SSH disconnection
launch = subprocess.run(
    ["ssh", "-o", "StrictHostKeyChecking=no", f"cc@{FLOATING_IP}",
     f"tmux new-session -d -s train '{train_cmd}' && echo 'launched'"],
    capture_output=True, text=True
)
if launch.returncode == 0:
    print(f"Training launched in tmux session 'train' on {FLOATING_IP}")
    print(f"  Monitor: ssh cc@{FLOATING_IP}  →  tmux attach -t train")
else:
    print(f"Launch failed: {launch.stderr.strip()}")
    print(f"Manual command: ssh cc@{FLOATING_IP}")
    print(f"  {train_cmd}")

## 7. Monitor Training Progress

In [ ]:
import re

# Poll training log every 60s and display latest step
# LeRobot logs lines like: "step=1000 loss=0.043 ..."
LOG_PATH = f"~/lerobot/outputs/train/{POLICY}_{DATASET_SLUG}/train.log"
TOTAL_STEPS = 100000  # adjust to your training config
POLL_INTERVAL = 60
POLLS = 10  # check 10 times then stop — re-run cell to continue monitoring

with tqdm(total=TOTAL_STEPS, unit="step", desc="Training") as pbar:
    last_step = 0
    for _ in range(POLLS):
        log_tail = subprocess.run(
            ["ssh", "-o", "StrictHostKeyChecking=no", f"cc@{FLOATING_IP}",
             f"tail -n 5 {LOG_PATH} 2>/dev/null || echo 'log not found'"],
            capture_output=True, text=True
        )
        log_out = log_tail.stdout.strip()
        # Parse current step from log
        steps = re.findall(r'step=(\d+)', log_out)
        if steps:
            current_step = int(steps[-1])
            pbar.update(current_step - last_step)
            last_step = current_step
        print(f"  {log_out.splitlines()[-1] if log_out else 'waiting...'}")
        if last_step >= TOTAL_STEPS:
            print("Training complete!")
            break
        time.sleep(POLL_INTERVAL)

_bench["timings"]["training_monitor_s"] = round(time.monotonic() - _t_train, 1)

## 8. Upload Checkpoint and Fetch to Pi

In [ ]:
_t_upload = time.monotonic()

# Upload from training node to HF Hub
upload_result = subprocess.run(
    ["ssh", "-o", "StrictHostKeyChecking=no", f"cc@{FLOATING_IP}",
     f"HF_TOKEN={HF_TOKEN} huggingface-cli upload {MODEL_REPO} "
     f"~/lerobot/outputs/train/{POLICY}_{DATASET_SLUG}/checkpoints/last/ ."],
    capture_output=True, text=True
)
if upload_result.returncode == 0:
    print(f"Checkpoint uploaded: https://huggingface.co/{MODEL_REPO}")
else:
    print(f"Upload failed: {upload_result.stderr.strip()}")

_bench["timings"]["upload_s"] = round(time.monotonic() - _t_upload, 1)

# Fetch to Pi
print()
print("Fetch checkpoint on Pi (inside Docker):")
print(f"  bash /app/scripts/fetch_checkpoint.sh {MODEL_REPO}")

## Benchmark: Save Timing

In [ ]:
_bench["timings"]["total_s"] = round(time.monotonic() - _t0, 1)
_bench["completed_at"] = datetime.utcnow().isoformat()

results_dir = Path("..") / "bench" / "results"
results_dir.mkdir(exist_ok=True)
ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
bench_path = results_dir / f"02_lerobot_{ts}.json"
bench_path.write_text(json.dumps(_bench, indent=2))
print(json.dumps(_bench, indent=2))
print(f"\nBenchmark saved: {bench_path}")